In [38]:
library_load <- suppressMessages(
    
    list(
        
        # Seurat 
        library(Seurat), 
        
        # Data 
        library(tidyverse), 
        library(data.table), 

        library(lme4), 
        library(emmeans), 
        
        # Plotting 
        library(patchwork), 
        
        # Pyhton compatibility
        library(reticulate)
        
    )
    
)

In [39]:
# Configure reticulate 
use_condaenv(condaenv='p.3.10.16-FD20250213HSCT', conda="/nobackup/lab_stary/fdeckert/bin/miniconda3/bin/conda", required=NULL)
py_config()

python:         /nobackup/lab_stary/fdeckert/bin/miniconda3/envs/p.3.10.16-FD20250213HSCT/bin/python
libpython:      /nobackup/lab_stary/fdeckert/bin/miniconda3/envs/p.3.10.16-FD20250213HSCT/lib/libpython3.10.so
pythonhome:     /nobackup/lab_stary/fdeckert/bin/miniconda3/envs/p.3.10.16-FD20250213HSCT:/nobackup/lab_stary/fdeckert/bin/miniconda3/envs/p.3.10.16-FD20250213HSCT
version:        3.10.16 | packaged by conda-forge | (main, Apr  8 2025, 20:53:32) [GCC 13.3.0]
numpy:          /nobackup/lab_stary/fdeckert/bin/miniconda3/envs/p.3.10.16-FD20250213HSCT/lib/python3.10/site-packages/numpy
numpy_version:  2.2.6

NOTE: Python version was forced by use_python() function

In [40]:
options(warn=-1)

In [41]:
random_seed <- 42
set.seed(random_seed)

In [42]:
# Set working directory to project root
setwd("/research/lab_stary/fdeckert/FD20250213HSCT")

In [43]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Import 

In [44]:
so <- readRDS("data/object/pp.rds")

# Cell type counts

In [69]:
df <- so@meta.data %>% dplyr::filter(hto_demux_class=="Blood")
counts_blood <- df %>%
    group_by(celltype_low) %>%
    group_split() %>%
    setNames(df %>% group_by(celltype_low) %>% group_keys() %>% pull(celltype_low)) %>%
    lapply(function(sub) {
        sub %>%
        count(patient_id, time_point) %>%
        pivot_wider(names_from = time_point, values_from = n, values_fill = 0)
  }
          )
names(counts_blood) <- gsub("\\+", "pos", names(counts_blood) )
names(counts_blood) <- gsub("\\.", "", make.names(names(counts_blood)))

In [70]:
openxlsx::write.xlsx(counts_blood, "result/statistics/celltype_low_count_blood.xlsx", colNames=TRUE, rowNames=TRUE)

In [71]:
df <- so@meta.data %>% dplyr::filter(hto_demux_class=="Skin")
counts_skin <- df %>%
    group_by(celltype_low) %>%
    group_split() %>%
    setNames(df %>% group_by(celltype_low) %>% group_keys() %>% pull(celltype_low)) %>%
    lapply(function(sub) {
        sub %>%
        count(patient_id, time_point) %>%
        pivot_wider(names_from = time_point, values_from = n, values_fill = 0)
  }
          )
names(counts_skin) <- gsub("\\+", "pos", names(counts_skin))
names(counts_skin) <- gsub("\\.", "", make.names(names(counts_skin)))

In [72]:
openxlsx::write.xlsx(counts_skin, "result/statistics/celltype_low_count_skin.xlsx", colNames=TRUE, rowNames=TRUE)

# Cell count check 

In [49]:
# Total cell genes vs cells 
nrow(so)
ncol(so)

[1] 36601

[1] 84447

# Genotype checks 

In [50]:
# Genotype statistics 
vireo <- read.csv("result/genotype/genotype.csv", row.names=1)
vireo <- vireo[colnames(so), ]
vireo <- vireo[vireo$vireo_patient_id!="Patient_2", ]

In [51]:
100*table(vireo$vireo_genotype_class)/sum(table(vireo$vireo_genotype_class))


     Donor    doublet       Host unassigned 
33.1468788  0.9204406 63.5973072  2.3353733 

In [52]:
# Genotype time points  
vireo_1 <- vireo[colnames(so[, so$time_point %in% c("Baseline", "Tx")]), ]
table(vireo_1$vireo_genotype_best_class)
sum(table(vireo_1$vireo_genotype_best_class))


Donor  Host 
   19 30829 

[1] 30848

In [53]:
# Genotype non-hematopoetic cells  
vireo_2 <- vireo[colnames(so[, so$celltype_domain %in% c("'Non-hematopoetic'")]), ]
table(vireo_2$vireo_genotype_best_class)
sum(table(vireo_2$vireo_genotype_best_class))


Donor  Host 
   77 26707 

[1] 26784

In [54]:
# Celltype low genotype counts skin 
mat <- so@meta.data %>% dplyr::filter(hto_demux_class=="Skin") %>%
    dplyr::group_by(patient_id, time_point, celltype_low, genotype_class) %>% dplyr::summarise(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::mutate(time_point=paste0(time_point, "_", genotype_class)) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=)) %>% 
    as.data.frame() 

mat <- split(mat, f=mat$celltype_low)

mat <- lapply(mat, function(x) {

    x <- x %>% 
        tidyr::complete(patient_id=levels(patient_id), time_point=levels(time_point)) %>%
        tidyr::pivot_wider(id_cols=patient_id, names_from=time_point, values_from=n)

    rownames(x) <- x$patient_id
    x <- x[paste0("Patient_", 1:12), c("patient_id", "Baseline_Host", "Tx_Host", "D14_Host", "D14_Donor", "D100_Host", "D100_Donor", "GVHD_Host", "GVHD_Donor")]

    return(x)
    
}
      )

names(mat) <- str_replace_all(names(mat) , "\\+", "plus") %>% make.names()
openxlsx::write.xlsx(mat, "result/statistics/celltype_low_genotype_count_skin.xlsx", colNames=TRUE, rowNames=TRUE)

`summarise()` has grouped output by 'patient_id', 'time_point', 'celltype_low'.
You can override using the `.groups` argument.


In [55]:
# Celltype low genotype counts skin 
mat <- so@meta.data %>% dplyr::filter(hto_demux_class=="Blood") %>%
    dplyr::group_by(patient_id, time_point, celltype_low, genotype_class) %>% dplyr::summarise(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::mutate(time_point=paste0(time_point, "_", genotype_class)) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=)) %>% 
    as.data.frame() 

mat <- split(mat, f=mat$celltype_low)

mat <- lapply(mat, function(x) {

    x <- x %>% 
        tidyr::complete(patient_id=levels(patient_id), time_point=levels(time_point)) %>%
        tidyr::pivot_wider(id_cols=patient_id, names_from=time_point, values_from=n)

    rownames(x) <- x$patient_id
    x <- x[paste0("Patient_", 1:12), c("patient_id", "Baseline_Host", "Tx_Host", "D14_Host", "D14_Donor", "D100_Host", "D100_Donor", "GVHD_Host", "GVHD_Donor")]

    return(x)
    
}
      )

names(mat) <- str_replace_all(names(mat) , "\\+", "plus") %>% make.names()
openxlsx::write.xlsx(mat, "result/statistics/celltype_low_genotype_count_blood.xlsx", colNames=TRUE, rowNames=TRUE)

`summarise()` has grouped output by 'patient_id', 'time_point', 'celltype_low'.
You can override using the `.groups` argument.


# Statistics 

In [56]:
data <- read.csv("result/statistics/tc17_time_point_data.csv", row.names=1) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=names(color$time_point))) %>% 
    dplyr::mutate(
        ratio_cells=n_cells/n_cells_total,
        arcsine_cells=asin(sqrt(n_cells/n_cells_total))
    )

In [57]:
model <- lmer(ratio_cells~time_point + (1|patient_id), data=data %>% dplyr::filter(hto_demux_class=="Skin" & data$t_cell_subset=="Tc17hi"))
openxlsx::write.xlsx(emmeans(model, pairwise~time_point, adjust="sidak"), "result/statistics/tc17_time_point_skin_tc17hi.xlsx", colNames=TRUE, rowNames=TRUE)

In [58]:
model <- lmer(ratio_cells~time_point + (1|patient_id), data=data %>% dplyr::filter(hto_demux_class=="Skin" & data$t_cell_subset=="Tc17lo"))
openxlsx::write.xlsx(emmeans(model, pairwise~time_point, adjust="sidak"), "result/statistics/tc17_time_point_skin_tc17lo.xlsx", colNames=TRUE, rowNames=TRUE)

In [59]:
data <- read.csv("result/statistics/th17_time_point_data.csv", row.names=1) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=names(color$time_point))) %>% 
    dplyr::mutate(
        ratio_cells=n_cells/n_cells_total,
        arcsine_cells=asin(sqrt(n_cells/n_cells_total))
    )

In [60]:
model <- lmer(ratio_cells~time_point + (1|patient_id), data=data %>% dplyr::filter(hto_demux_class=="Skin" & data$t_cell_subset=="Th17hi"))
openxlsx::write.xlsx(emmeans(model, pairwise~time_point, adjust="sidak"), "result/statistics/tc17_time_point_skin_th17hi.xlsx", colNames=TRUE, rowNames=TRUE)

In [61]:
model <- lmer(ratio_cells~time_point + (1|patient_id), data=data %>% dplyr::filter(hto_demux_class=="Skin" & data$t_cell_subset=="Th17lo"))
openxlsx::write.xlsx(emmeans(model, pairwise~time_point, adjust="sidak"), "result/statistics/tc17_time_point_skin_th17lo.xlsx", colNames=TRUE, rowNames=TRUE)

# abTCR analysis

In [62]:
tcr_id_blood <- so@meta.data %>% dplyr::filter(hto_demux_class=="Blood") %>% dplyr::pull(clone_id) %>% na.omit()
tcr_id_skin <- so@meta.data %>% dplyr::filter(hto_demux_class=="Skin") %>% dplyr::pull(clone_id) %>% na.omit()

In [63]:
tcr_id_shared <- intersect(tcr_id_blood, tcr_id_skin)

In [64]:
mat_1 <- so@meta.data %>% dplyr::filter(clone_id %in% tcr_id_shared) %>% dplyr::select(patient_id, clone_id) %>% 
    dplyr::group_by(patient_id) %>% dplyr::mutate(n_tcr=length(unique(clone_id)), n_cells_tcr_total=n()) %>% 
    dplyr::group_by(patient_id, clone_id) %>% dplyr::mutate(n_cells_tcr=n()) %>% 
    dplyr::group_by(patient_id, n_tcr, n_cells_tcr_total) %>% dplyr::summarise(n_cells_tcr_mean=mean(n_cells_tcr))

`summarise()` has grouped output by 'patient_id', 'n_tcr'. You can override
using the `.groups` argument.


In [65]:
mat_2 <- so@meta.data %>% dplyr::filter(clone_id %in% tcr_id_shared & genotype_class=="Host") %>% dplyr::select(patient_id, clone_id) %>% 
    dplyr::group_by(patient_id) %>% dplyr::mutate(n_tcr=length(unique(clone_id)), n_cells_tcr_total=n()) %>% 
    dplyr::group_by(patient_id, clone_id) %>% dplyr::mutate(n_cells_tcr=n()) %>% 
    dplyr::group_by(patient_id, n_tcr, n_cells_tcr_total) %>% dplyr::summarise(n_cells_tcr_mean=mean(n_cells_tcr))

`summarise()` has grouped output by 'patient_id', 'n_tcr'. You can override
using the `.groups` argument.


In [66]:
mat_3 <- so@meta.data %>% dplyr::filter(clone_id %in% tcr_id_shared & genotype_class=="Donor") %>% dplyr::select(patient_id, clone_id) %>% 
    dplyr::group_by(patient_id) %>% dplyr::mutate(n_tcr=length(unique(clone_id)), n_cells_tcr_total=n()) %>% 
    dplyr::group_by(patient_id, clone_id) %>% dplyr::mutate(n_cells_tcr=n()) %>% 
    dplyr::group_by(patient_id, n_tcr, n_cells_tcr_total) %>% dplyr::summarise(n_cells_tcr_mean=mean(n_cells_tcr))

`summarise()` has grouped output by 'patient_id', 'n_tcr'. You can override
using the `.groups` argument.


In [67]:
openxlsx::write.xlsx(list(all=mat_1, host=mat_2, donor=mat_2), "result/statistics/shared_tcr_blood_skin.xlsx", colNames=TRUE, rowNames=TRUE)